In [7]:
import pandas as pd 
from transformers import T5Tokenizer, Trainer, T5ForConditionalGeneration, TrainingArguments 

In [8]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [9]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [12]:
train_data.shape

(14732, 3)

In [15]:
# RANDOM SAMPLING 

train_data = train_data.sample(n = 4000, random_state = 42).reset_index(drop = True)
val_data = val_data.sample(n = 500, random_state = 42).reset_index(drop = True)

In [16]:
train_data.shape

(4000, 3)

## DATA PRE-PROCESSING 

In [17]:
import re 

def clean_data(text):
    text = re.sub(r"\r\n", " ", text) # replacing lines
    text = re.sub(r"\s+", " ", text) # replacing spaces
    text = re.sub(r"<.*?>", " ", text) # replacing html tags 
    text = text.strip().lower()

    return text

In [18]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [20]:
train_data["dialogue"][0]

"david: hey alex, do you still teach english? alex: hi!!! long time no hear :) yeah, i do, what's up? david: look, my boss is looking for someone who could teach him, so i thought about you. alex: sure thing! what level is he at now? david: hm... hard to tell, haven't asked ;) i think he just wants to be fluent, so he probably needs some to have conversations with. alex: no problem, the thing is i only have mondays and wednesdays off... david: oh my, that's perfect! he's quite a busy guy, but he said wednesday sounds fine. alex: great then! thanks for this! :) david: hahaha, no problem! so, can i text him your number? alex: of course :) david: he'll call you in the evening, we're still in the office. he's quite tech-savvy so he can also text you on what's app :d alex: hahaha, great! how are you by the way? i owe you a coffee for this! david: shut up, you owe me nothing, but i would happily go out, we have so much catching up alex: yeah, especially that job of yours! sounds quite cool, 

## TOKENIZATION 

In [22]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [45]:
# RAW DATA (INPUTS) => TOKENS 

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding = "max_length", max_length = 512, truncation = True)
    targets = tokenizer(data["summary"], padding = "max_length", max_length = 150, truncation = True)

    inputs["labels"] = targets["input_ids"]

    return inputs 

In [46]:
train_dataset = train_data.apply(tokenize, axis = 1).tolist()
val_dataset = val_data.apply(tokenize, axis = 1).tolist()

In [47]:
train_dataset[0]

{'input_ids': [836, 6961, 10, 3, 13133, 1240, 226, 6, 103, 25, 341, 3884, 22269, 58, 1240, 226, 10, 7102, 3158, 307, 97, 150, 1616, 3, 10, 61, 17945, 6, 3, 23, 103, 6, 125, 31, 7, 95, 58, 836, 6961, 10, 320, 6, 82, 7930, 19, 479, 21, 841, 113, 228, 3884, 376, 6, 78, 3, 23, 816, 81, 25, 5, 1240, 226, 10, 417, 589, 55, 125, 593, 19, 3, 88, 44, 230, 58, 836, 6961, 10, 3, 107, 51, 233, 614, 12, 817, 6, 43, 29, 31, 17, 1380, 3, 117, 61, 3, 23, 317, 3, 88, 131, 2746, 12, 36, 6720, 295, 6, 78, 3, 88, 1077, 523, 128, 12, 43, 9029, 28, 5, 1240, 226, 10, 150, 682, 6, 8, 589, 19, 3, 23, 163, 43, 1911, 1135, 7, 11, 62, 26, 1496, 1135, 7, 326, 233, 836, 6961, 10, 3, 32, 107, 82, 6, 24, 31, 7, 626, 55, 3, 88, 31, 7, 882, 3, 9, 3164, 4024, 6, 68, 3, 88, 243, 62, 26, 1496, 1135, 2993, 1399, 5, 1240, 226, 10, 248, 258, 55, 2049, 21, 48, 55, 3, 10, 61, 836, 6961, 10, 4244, 1024, 1024, 6, 150, 682, 55, 78, 6, 54, 3, 23, 1499, 376, 39, 381, 58, 1240, 226, 10, 13, 503, 3, 10, 61, 836, 6961, 10, 3, 88, 31, 

In [34]:
len(train_dataset[0]["attention_mask"])

512

## WORKING WITH THE MODEL

In [49]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [54]:
# selecting the device 

import torch 
import torch.nn as nn 

# if torch.backends.mps.is_available():
#     device = torch.device("mps")

# elif torch.cuda.is_available():
#     device = torch.device("cuda")

# else : 
device = torch.device("cpu")

print("device : ", device)
model.to(device)

device :  mps


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [56]:
# DEFINING TRAINING ARGUMENTS 

training_args = TrainingArguments(
    output_dir = "./results", 
    
    num_train_epochs = 6, 
    weight_decay = 0.01, 

    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8, 

    eval_strategy = "epoch", 
    save_strategy = "epoch", 

    warmup_steps = 500
)

In [58]:
# CREATING TRAINER => HELPS IN TRAINING CODE SIMPLIFICATION

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

In [59]:
# TRAINING THE MODEL

trainer.train()

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,15.338657,17.318377
2,15.360197,17.318377
3,15.363762,17.318377
4,15.366343,17.318377
5,15.352672,17.318377
6,15.314235,17.318377


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=15.34931103515625, metrics={'train_runtime': 37153.5491, 'train_samples_per_second': 0.646, 'train_steps_per_second': 0.081, 'total_flos': 3248203235328000.0, 'train_loss': 15.34931103515625, 'epoch': 6.0})

In [60]:
# SAVING THE MODEL 

model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_tokenizer")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_tokenizer/tokenizer_config.json', './saved_tokenizer/tokenizer.json')

In [62]:
# LOADING SAVE MODEL AND TOKENIZER 

model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

## TESTING THE TEXT SUMMARIZER 

In [75]:
def summarize_dialogue(dialogue):
    
    dialogue = clean_data(dialogue) # clean

    inputs = tokenizer(dialogue, 
        padding = "max_length",
        max_length = 512, 
        truncation = True, 
        return_tensors = "pt").to(device) # tokenize

    model.to(device)
    targets = model.generate(
        input_ids = inputs["input_ids"], 
        attention_mask = inputs["attention_mask"], 
        max_length = 150, 
        num_beams = 4, # generate 4 summary sequences and after comparing, return the best 
        early_stopping = True
    )

    summary = tokenizer.decode(targets[0], skip_special_tokens = True) # generating summary 
    return summary 

In [76]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)
print("Summary :", summary)

Summary : experts highlight the importance of responsible ai development, including data privacy, security, and long-term societal impact. ensuring fairness and transparency is becoming a key area of research.
